In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import TensorDataset, DataLoader
import torchmetrics
import numpy as np


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [3]:
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(root='datasets', train=True, download=False, transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(root='datasets', train=False, download=False, transform=toTensor)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [55_000, 5_000])

In [4]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=False)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [5]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )

    def forward(self, X):
        return self.mlp(X)


In [6]:
def train(model, device, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}; Loss: {mean_loss:.4f}")


In [33]:
def train2(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, device, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history


In [8]:
#torch.manual_seed(42)
#learning_rate = 0.01
#model = nn.Sequential(
#    nn.Linear(n_features, 50), nn.ReLU(),
#    nn.Linear(50, 40), nn.ReLU(),
#    nn.Linear(40, 30), nn.ReLU(),
#    nn.Linear(30, 1)
#)
#model = model.to(device)
#optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0)
#mse = nn.MSELoss()
#rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
#history = train2(model, optimizer, mse, rmse, train_loader, valid_loader,
#                 n_epochs)

# Since we compute the training metric
#plt.plot(np.arange(n_epochs) + 0.5, history["train_metrics"], ".--",
#         label="Training")
#plt.plot(np.arange(n_epochs) + 1.0, history["valid_metrics"], ".-",
#         label="Validation")
#plt.xlabel("Epoch")
#plt.ylabel("RMSE")
#plt.grid()
#plt.title("Learning curves")
#plt.axis([0.5, 20, 0.4, 1.0])
#plt.legend()
#plt.show()

In [9]:
def evaluate(model, device, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, Y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
    return aggregate_fn(torch.stack(metrics))
 

In [32]:
def evaluate_tm(model, device, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch) # Update metric at each iteration
    return metric.compute() # Compute the final result at the end


In [11]:
torch.manual_seed(42)
model = ImageClassifier(n_inputs=28*28, n_hidden1=300, n_hidden2=100, n_classes=10)
xentropy = nn.CrossEntropyLoss()

In [12]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to(device)

In [13]:
learning_rate = 0.1
n_epochs = 50
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
train(model, device, optimizer, xentropy, train_loader, n_epochs)

Epoch 1/50; Loss: 0.6029
Epoch 2/50; Loss: 0.4041
Epoch 3/50; Loss: 0.3594
Epoch 4/50; Loss: 0.3305
Epoch 5/50; Loss: 0.3096
Epoch 6/50; Loss: 0.2919
Epoch 7/50; Loss: 0.2771
Epoch 8/50; Loss: 0.2638
Epoch 9/50; Loss: 0.2516
Epoch 10/50; Loss: 0.2434
Epoch 11/50; Loss: 0.2329
Epoch 12/50; Loss: 0.2231
Epoch 13/50; Loss: 0.2149
Epoch 14/50; Loss: 0.2060
Epoch 15/50; Loss: 0.2002
Epoch 16/50; Loss: 0.1903
Epoch 17/50; Loss: 0.1839
Epoch 18/50; Loss: 0.1785
Epoch 19/50; Loss: 0.1738
Epoch 20/50; Loss: 0.1703
Epoch 21/50; Loss: 0.1626
Epoch 22/50; Loss: 0.1564
Epoch 23/50; Loss: 0.1540
Epoch 24/50; Loss: 0.1472
Epoch 25/50; Loss: 0.1427
Epoch 26/50; Loss: 0.1390
Epoch 27/50; Loss: 0.1381
Epoch 28/50; Loss: 0.1358
Epoch 29/50; Loss: 0.1297
Epoch 30/50; Loss: 0.1232
Epoch 31/50; Loss: 0.1239
Epoch 32/50; Loss: 0.1212
Epoch 33/50; Loss: 0.1236
Epoch 34/50; Loss: 0.1137
Epoch 35/50; Loss: 0.1131
Epoch 36/50; Loss: 0.1085
Epoch 37/50; Loss: 0.1056
Epoch 38/50; Loss: 0.1022
Epoch 39/50; Loss: 0.

In [14]:
evaluate_tm(model, device, valid_loader, accuracy)

tensor(0.8786)

In [15]:
model.eval()
X_new, y_new = next(iter(valid_loader))
X_new = X_new[:3].to(device)
with torch.no_grad():
    y_pred_logits = model(X_new)
y_pred_logits

tensor([[ -6.4478,   1.6428,  -2.0622,  -6.4511,  -5.6654,   6.4118,  -7.2326,
          21.3954,  -6.6998,   6.8317],
        [ -6.3034,   0.9305,  10.7111,  -0.4232,  20.5535, -12.1042,   4.6138,
         -10.2643,  -0.8547,  -8.8095],
        [  1.2677,  -4.0814,   9.0591,   1.1005,   5.0175,  -6.0332,  10.1575,
          -5.4626,  -6.4725,  -4.8443]])

In [16]:
print(type(y_pred_logits))
y_pred = y_pred_logits.argmax(dim=1)
print(type(y_pred))
y_pred


<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([7, 4, 6])

In [17]:
[train_and_valid_data.classes[index] for index in y_pred]

['Sneaker', 'Coat', 'Shirt']

In [18]:
import torch.nn.functional as F

In [19]:
y_proba = F.softmax(y_pred_logits, dim=1)
y_proba.round(decimals=3)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.2490, 0.0000, 0.0040, 0.0000, 0.7470, 0.0000, 0.0000,
         0.0000]])

In [20]:
y_top4_logits, y_top4_indices = torch.topk(y_pred_logits, k=4, dim=1)
y_top4_probas = F.softmax(y_top4_logits, dim=1)
y_top4_probas.round(decimals=3)

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000],
        [0.7470, 0.2490, 0.0040, 0.0000]])

In [21]:
y_top4_indices

tensor([[7, 9, 5, 1],
        [4, 2, 6, 1],
        [6, 2, 4, 0]])

In [22]:
import optuna

In [27]:
def objective(trial, train_loader, valid_loader):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)
    model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden,
                            n_hidden2=n_hidden, n_classes=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
    accuracy = accuracy.to(device)
    best_validation_accuracy = 0.0
    for epoch in range(n_epochs):
        history = train2(model, optimizer, xentropy, accuracy, train_loader,
                         valid_loader, n_epochs=1)
        validation_accuracy = max(history["valid_metrics"])
        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
        trial.report(validation_accuracy, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_validation_accuracy

In [28]:
objective_with_data = lambda trial: objective(
    trial, train_loader=train_loader, valid_loader=valid_loader)

In [29]:
from functools import partial

objective_with_data = partial(objective, train_loader=train_loader,
                              valid_loader=valid_loader)

In [34]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner()
study = optuna.create_study(direction="maximize", sampler=sampler,
                            pruner=pruner)
study.optimize(objective_with_data, n_trials=20)

[I 2026-10-01 23:02:25,572] A new study created in memory with name: no-name-d16bce7f-7007-49db-b046-e053750f124d


Epoch 1/1, train loss: 2.2769, train metric: 0.1435, valid metric: 0.1902
Epoch 1/1, train loss: 2.2093, train metric: 0.2713, valid metric: 0.3540
Epoch 1/1, train loss: 2.1164, train metric: 0.4102, valid metric: 0.4542
Epoch 1/1, train loss: 1.9775, train metric: 0.5129, valid metric: 0.5548
Epoch 1/1, train loss: 1.7866, train metric: 0.5864, valid metric: 0.5994
Epoch 1/1, train loss: 1.5774, train metric: 0.6183, valid metric: 0.6182
Epoch 1/1, train loss: 1.3978, train metric: 0.6293, valid metric: 0.6264
Epoch 1/1, train loss: 1.2605, train metric: 0.6370, valid metric: 0.6340
Epoch 1/1, train loss: 1.1572, train metric: 0.6446, valid metric: 0.6384
Epoch 1/1, train loss: 1.0782, train metric: 0.6533, valid metric: 0.6482
Epoch 1/1, train loss: 1.0162, train metric: 0.6610, valid metric: 0.6578
Epoch 1/1, train loss: 0.9665, train metric: 0.6686, valid metric: 0.6644
Epoch 1/1, train loss: 0.9258, train metric: 0.6758, valid metric: 0.6708
Epoch 1/1, train loss: 0.8918, train m

[I 2026-10-01 23:17:18,637] Trial 0 finished with value: 0.8014000058174133 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.8014000058174133.


Epoch 1/1, train loss: 0.5402, train metric: 0.8157, valid metric: 0.8014
Epoch 1/1, train loss: 1.1653, train metric: 0.6080, valid metric: 0.7446
Epoch 1/1, train loss: 0.6223, train metric: 0.7823, valid metric: 0.8028
Epoch 1/1, train loss: 0.5250, train metric: 0.8167, valid metric: 0.8154
Epoch 1/1, train loss: 0.4824, train metric: 0.8296, valid metric: 0.8256
Epoch 1/1, train loss: 0.4570, train metric: 0.8379, valid metric: 0.8310
Epoch 1/1, train loss: 0.4384, train metric: 0.8454, valid metric: 0.8372
Epoch 1/1, train loss: 0.4233, train metric: 0.8505, valid metric: 0.8422
Epoch 1/1, train loss: 0.4101, train metric: 0.8546, valid metric: 0.8460
Epoch 1/1, train loss: 0.3985, train metric: 0.8589, valid metric: 0.8482
Epoch 1/1, train loss: 0.3884, train metric: 0.8622, valid metric: 0.8504
Epoch 1/1, train loss: 0.3789, train metric: 0.8650, valid metric: 0.8530
Epoch 1/1, train loss: 0.3704, train metric: 0.8677, valid metric: 0.8544
Epoch 1/1, train loss: 0.3625, train m

[I 2026-10-01 23:31:47,697] Trial 1 finished with value: 0.8841999769210815 and parameters: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}. Best is trial 1 with value: 0.8841999769210815.


...

Epoch 1/1, train loss: 0.1012, train metric: 0.9661, valid metric: 0.8816


[I 2026-10-02 02:14:45,720] Trial 18 pruned. 


Epoch 1/1, train loss: 2.0528, train metric: 0.4009, valid metric: 0.5378
Epoch 1/1, train loss: 0.6197, train metric: 0.7745, valid metric: 0.8246
Epoch 1/1, train loss: 0.4131, train metric: 0.8494, valid metric: 0.8492
Epoch 1/1, train loss: 0.3677, train metric: 0.8646, valid metric: 0.8586
Epoch 1/1, train loss: 0.3386, train metric: 0.8749, valid metric: 0.8634
Epoch 1/1, train loss: 0.3168, train metric: 0.8826, valid metric: 0.8666
Epoch 1/1, train loss: 0.2991, train metric: 0.8901, valid metric: 0.8720
Epoch 1/1, train loss: 0.2840, train metric: 0.8945, valid metric: 0.8742
Epoch 1/1, train loss: 0.2722, train metric: 0.8986, valid metric: 0.8750
Epoch 1/1, train loss: 0.2591, train metric: 0.9034, valid metric: 0.8736
Epoch 1/1, train loss: 0.2476, train metric: 0.9086, valid metric: 0.8762
Epoch 1/1, train loss: 0.2383, train metric: 0.9117, valid metric: 0.8756
Epoch 1/1, train loss: 0.2290, train metric: 0.9159, valid metric: 0.8778
Epoch 1/1, train loss: 0.2202, train m

[I 2026-10-02 02:27:34,758] Trial 19 finished with value: 0.8838000297546387 and parameters: {'learning_rate': 0.07631084095715764, 'n_hidden': 193}. Best is trial 11 with value: 0.8866000175476074.


Epoch 1/1, train loss: 0.0757, train metric: 0.9721, valid metric: 0.8724
